# Benchmarking Comparison

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 26.01.2025
- Date of Last Modification: 17.02.2025 (by Sebastian Birk; sb75@sanger.ac.uk)

- Run this notebook in the nichejepa conda environment, installable from ```../../envs/nichejepa.yaml```.

## 1. Setup

### 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import os
import warnings
from collections import OrderedDict

import anndata as ad
import pandas as pd
import scanpy as sc
from matplotlib.lines import Line2D

from analysis_utils import *
from benchmarking_utils import max_overlap_plot

### 1.2 Configure Paths

In [ ]:
st_data_folder_path = '/lustre/scratch126/cellgen/team361/DATASETS'
st_data_silver_folder_path = f'{st_data_folder_path}/silver'
artifacts_folder_path = '/lustre/scratch126/cellgen/team361/sb75/nichejepa-reproducibility/artifacts'
benchmarking_artifacts_folder_path = f'{artifacts_folder_path}/benchmarking'

### 1.3 Define Config

### 1.4 Run Notebook Setup

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{benchmarking_artifacts_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [ ]:
os.makedirs(f'{benchmarking_artifacts_folder_path}/figures', exist_ok=True)
os.makedirs(f'{benchmarking_artifacts_folder_path}/results', exist_ok=True)

## 2. Data Visualization

### 2.1 chl59-8b_1p (NanoString CosMx Human NSCLC)

In [ ]:
# Define dataset params
dataset_name = 'chl59-8b_1p'
batches = ['Lung9_Rep1',
           'Lung9_Rep2',
           'Lung5_Rep1',
           'Lung5_Rep2',
           'Lung6',
           'Lung12']
niche_key = 'niche'
batch_key = 'batch'
spot_sizes = [80 * 0.18] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)

            # Filter cells without niche labels
            adata = adata[~adata.obs[niche_key].isna()]

            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.18 # conversion rate from https://nanostring-public-share.s3.us-west-2.amazonaws.com/SMI-Compressed/SMI-ReadMe.html

In [ ]:
adata.obs[niche_key].unique()

In [ ]:
# Plot ground truth niches

# Label niches (based on NicheCompass reproducibility repo)
label2niche = {
    '0': '1- Tumor (stroma border)',
    '2': '2- Tumor interior',
    '5': '3- Tumor (neutrophil border)',
    '7': '4- Tumor interior',
    '11': '5- Infiltrated tumor',
    '3': '6- Tumor-infiltrating neutrophils',
    '4': '7- Neutrophil expansion',
    '1': '8- Stroma',
    '6': '9- Plasmablast rich stroma',
    '9': '10- Lymphoid rich stroma',
    '8': '11- Lymphoid aggregates',
    '10': '12- Macrophage rich stroma'
}

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_groundtruth_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=niche_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

### 2.2 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

In [ ]:
# Define dataset params
dataset_name = 'chr78-11b_1p'
batches = ['CM007',
           'CM008',
           'CM009',
           'CM011',
           'CM013',
           'CM015',
           'CM016',
           'CM017',
           'CM018',
           'CM019',
           'CM020']
niche_key = 'broad_anatomy'
batch_key = 'batch'
spot_sizes = [240 * 0.093] * len(batches)

In [ ]:
# Load adata batches and concatenate for inference
adata_list = []

for subdir, _, files in os.walk(f'{st_data_silver_folder_path}/{dataset_name}'):
    for file_idx, file in enumerate(files):
        if file.endswith('.h5ad') & any(batch in file for batch in batches):
            file_path = os.path.join(subdir, file)
            adata = sc.read_h5ad(file_path)
            adata.obs['batch'] = adata.obs['cell_id'].apply(lambda x: x.split('_')[0] + x.split('_')[1])
            adata_list.append(adata)

adata = ad.concat(adata_list, join='outer', index_unique=None)
adata.obsm['spatial'] = adata.obsm['spatial'] * 0.093 # conversion rate from Valentina

In [ ]:
# Combine regions
adata.obs[niche_key] = adata.obs[niche_key].apply(lambda x: x.replace('Urogenital Sinus', 'UrogenitalSinus'))

In [ ]:
# Plot ground truth niches

# Label niches
label2niche = {
    '0': 'Adrenal',
    '1': 'Cervix',
    '2': 'Ducts',
    '3': 'Epididymis',
    '4': 'Epoophron',
    '5': 'External Genitalia',
    '6': 'FallopianTube',
    '7': 'Genital Tubercle',
    '8': 'Gonad',
    '9': 'LabioScrotal Swelling',
    '10': 'LowerTract',
    '11': 'Nephros',
    '12': 'Testis',
    '13': 'Unassigned',
    '14': 'UpperVagina',
    '15': 'Urogenital Sinus',
    '16': 'UrogenitalSinus',
    '17': 'Uterus',
    '18': 'VaginalPlate',
    '19': 'Vas Deferens',
    '20': 'Background', # < 100 cells
}

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

batch_ids = adata.obs[batch_key].unique().tolist()
for i, batch_id in enumerate(batch_ids):
    adata_batch = adata[adata.obs[batch_key] == batch_id]
    file_name = f'{dataset_name}_{batch_id}_groundtruth_niches.svg'
    sc.pl.spatial(adata_batch,
                  color=niche_key,
                  palette=pred_palette,
                  spot_size=spot_sizes[i],
                  save=file_name)

### 2.3 xhs49-1b_1p (Xenium Human Melanoma)

In [ ]:
dataset_name = 'xhs49-1b_1p'
niche_key = 'niche_name'
spot_size = 20

In [ ]:
adata = sc.read_h5ad(f'/lustre/scratch126/cellgen/team361/DATASETS/silver/{dataset_name}/Xenium_Prime_Human_Skin_FFPE_outs.h5ad')

# Filter cells without niche labels
adata = adata[~adata.obs['niche_name'].isna()]

In [ ]:
adata.obs['niche_name'].value_counts()

In [ ]:
# Plot ground truth niches

# Label niches
label2niche = {
    '0': 'Epidemis',
    '1': 'Superficial dermis',
    '2': 'Melanoma+infiltrate',
    '3': 'Melanoma',
    '4': 'Immune infiltrate Thi',
    '5': 'Immune infiltrate Mac hi',
    '6': 'Macrophage_fat_region',
    '7': 'Melanoma+blood vessels'
}

pred_palette = {
    label2niche.get(k, k): v
    for k, v in custom_palette.items()
}

file_name = f'{dataset_name}_groundtruth_niches.svg'
sc.pl.spatial(adata,
              color=niche_key,
              palette=pred_palette,
              spot_size=spot_size,
              save=file_name)

## 3. Metric Comparisons

In [ ]:
nemo_49_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/xhs49-1b_1p_cellplm_metric_scores.csv')
nemo_59_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chl59-8b_1p_cellplm_metric_scores.csv')
nemo_78_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chr78-11b_1p_cellplm_metric_scores.csv')
cellplm_49_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/xhs49-1b_1p_cellplm_metric_scores.csv')
cellplm_59_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chl59-8b_1p_cellplm_metric_scores.csv')
cellplm_78_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chr78-11b_1p_cellplm_metric_scores.csv')
nicheformer_49_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/xhs49-1b_1p_cellplm_metric_scores.csv')
nicheformer_59_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chl59-8b_1p_cellplm_metric_scores.csv')
nicheformer_78_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chr78-11b_1p_cellplm_metric_scores.csv')
novae_49_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/xhs49-1b_1p_novae_metric_scores.csv')
novae_59_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chl59-8b_1p_novae_metric_scores.csv')
novae_78_df = pd.read_csv(f'{benchmarking_artifacts_folder_path}/results/chr78-11b_1p_novae_metric_scores.csv')

nemo_49_df['dataset'] = 'Xenium Human Melanoma'
nemo_59_df['dataset'] = 'NanoString CosMx Human NSCLC'
nemo_78_df['dataset'] = 'ISS Cartana Human Reproductive Tract'
cellplm_49_df['dataset'] = 'Xenium Human Melanoma'
cellplm_59_df['dataset'] = 'NanoString CosMx Human NSCLC'
cellplm_78_df['dataset'] = 'ISS Cartana Human Reproductive Tract'
nicheformer_49_df['dataset'] = 'Xenium Human Melanoma'
nicheformer_59_df['dataset'] = 'NanoString CosMx Human NSCLC'
nicheformer_78_df['dataset'] = 'ISS Cartana Human Reproductive Tract'
novae_49_df['dataset'] = 'Xenium Human Melanoma'
novae_59_df['dataset'] = 'NanoString CosMx Human NSCLC'
novae_78_df['dataset'] = 'ISS Cartana Human Reproductive Tract'

nemo_49_df['method'] = 'NEMO'
nemo_59_df['method'] = 'NEMO'
nemo_78_df['method'] = 'NEMO'
cellplm_49_df['method'] = 'CellPLM'
cellplm_59_df['method'] = 'CellPLM'
cellplm_78_df['method'] = 'CellPLM'
nicheformer_49_df['method'] = 'Nicheformer'
nicheformer_59_df['method'] = 'Nicheformer'
nicheformer_78_df['method'] = 'Nicheformer'
novae_49_df['method'] = 'Novae'
novae_59_df['method'] = 'Novae'
novae_78_df['method'] = 'Novae'

df = pd.concat([nemo_49_df,
                nemo_59_df,
                nemo_78_df,
                cellplm_49_df,
                cellplm_59_df,
                cellplm_78_df,
                nicheformer_49_df,
                nicheformer_59_df,
                nicheformer_78_df,
                novae_49_df,
                novae_59_df,
                novae_78_df],
               ignore_index=True)

In [ ]:
plt.rcParams.update({
    'font.size': 24,
    'axes.titlesize': 24,
    'axes.labelsize': 24,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'legend.fontsize': 24,
    'legend.title_fontsize': 24
})

# Set the aesthetic style of the plots
sns.set(style="whitegrid")

# Initialize the matplotlib figure with a specific size
plt.figure(figsize=(11, 6))  # Increased figure size for better visibility

# Define a color palette (optional: customize as needed)
palette = "Set2"  # You can choose other palettes like "Set1", "Set3", etc.

# Create the boxplot
ax = sns.boxplot(
    x='dataset',
    y='NMI',
    hue='method',
    data=df,
    palette=palette,
    dodge=True,
    showfliers=False  # Hide outliers for a cleaner look
)

# Overlay the scatter points with the same hue and palette
sns.stripplot(
    x='dataset',
    y='NMI',
    hue='method',
    data=df,
    dodge=True,
    jitter=True,
    alpha=0.7,
    size=8,  # Increased dot size
    marker='o',
    edgecolor='black',  # Changed to black edges
    linewidth=1,        # Add linewidth for better edge visibility
    palette=palette,    # Ensure the palette matches the boxplot
    ax=ax
)

# Handle the legend to avoid duplicates
handles, labels = ax.get_legend_handles_labels()
by_label = OrderedDict(zip(labels, handles))  # Remove duplicate labels

# Remove the existing legend
ax.legend_.remove()

# Create custom legend handles with desired properties
unique_methods = list(by_label.keys())
palette_colors = sns.color_palette(palette, n_colors=len(unique_methods))

custom_handles = [
    Line2D([0], [0], marker='o', color='w', label=method,
           markerfacecolor=palette_colors[i],
           markeredgecolor='black', markersize=10)
    for i, method in enumerate(unique_methods)
]

# Add the custom legend
ax.legend(handles=custom_handles, title='Method')

# Add titles and labels with appropriate font sizes
plt.title('NMI Distribution per Dataset and Method')
plt.xlabel('Dataset')
plt.ylabel('NMI')

# Adjust layout for better fit and prevent clipping of labels
plt.tight_layout()

# Display the plot
plt.savefig(f'{benchmarking_artifacts_folder_path}/figures/benchmarking_comparison.svg', dpi=300)
plt.show()

In [ ]:
# 1. Specify the dataset you want to plot
selected_dataset = 'NanoString CosMx Human NSCLC'  # Replace with your desired dataset name

# 2. Filter the DataFrame for the selected dataset
filtered_df = df[df['dataset'] == selected_dataset]

# 3. Update matplotlib parameters for consistent styling
plt.rcParams.update({
    'font.size': 24,
    'axes.titlesize': 24,
    'axes.labelsize': 24,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'legend.fontsize': 24,
    'legend.title_fontsize': 24
})

# Set the aesthetic style of the plots
sns.set(style="whitegrid")

# 4. Initialize the matplotlib figure with a specific size
plt.figure(figsize=(5, 4))  # Increased figure size for better visibility

# 5. Create the boxplot for the filtered dataset without hue and with customized box properties
ax = sns.boxplot(
    x='method',          # X-axis represents different methods
    y='NMI',
    data=filtered_df,
    dodge=False,         # No dodge since hue is removed
    showfliers=False,    # Hide outliers for a cleaner look
    width=0.6,           # Adjust box width as needed
    boxprops={
        'facecolor': 'lightgrey',  # Light grey fill for boxes
        'edgecolor': 'black',      # Black edges for boxes
        'linewidth': 1.5            # Thickness of box edges
    },
    medianprops={
        'color': 'black',          # Black median lines
        'linewidth': 2
    },
    whiskerprops={
        'color': 'grey',           # Grey whiskers
        'linewidth': 1.5
    },
    capprops={
        'color': 'grey',           # Grey caps
        'linewidth': 1.5
    }
)

# 6. Overlay the scatter points with black color
sns.stripplot(
    x='method',
    y='NMI',
    data=filtered_df,
    jitter=True,
    alpha=0.7,
    size=5,                  # Increased dot size
    marker='o',
    color='black',           # Black dots
    edgecolor='black',       # Black edges for dots
    linewidth=1,             # Add linewidth for better edge visibility
    ax=ax
)

# 7. Remove the existing legend since hue is not used
if ax.get_legend() is not None:
    ax.legend_.remove()

# 8. Add titles and labels with appropriate font sizes
plt.title(f'NMI Distribution for {selected_dataset}')
plt.xlabel('Method')
plt.ylabel('NMI')

# 9. (Optional) If you want to format the y-axis, you can uncomment and adjust the following lines
# ax.yaxis.set_major_formatter(
#     ticker.FuncFormatter(lambda x, pos: f"{x/1e6:.1f}M")
# )

# 10. Add vertical gridlines for better readability (optional)
ax.grid(
    axis='y',
    which='major',
    linestyle='--',  # Dash pattern: 5 points on, 10 points off
    color='grey',
    alpha=0.5
)

# 11. Adjust layout for better fit and prevent clipping of labels
plt.tight_layout()

# 12. Save the figure
plt.savefig(f'{benchmarking_artifacts_folder_path}/figures/benchmarking_comparison_{selected_dataset.replace(" ", "_")}.svg', dpi=300, bbox_inches='tight')

# 13. Display the plot
plt.show()